# Suzuki Autó Adatelemzés

**Exploratory Data Analysis (EDA) a Suzuki (Maruti Suzuki) használtautó-piacról**

Ez a notebook egy tipikus Kaggle autópiaci EDA felépítését és mélységét követi
(bevezetés → adattisztítás → egyváltozós elemzés → két-/többváltozós elemzés →
üzleti következtetések), egyetlen gyártóra, a **Suzukira** szabva.


## 1. Bevezetés

A Suzuki a világ egyik legnagyobb autógyártója, különösen erős a belépő szintű és
kompakt kategóriákban indiai vegyesvállalatán, a **Maruti Suzukin** keresztül, amely
piacvezető India személygépjármű-piacán.

**A projekt célja:** egy teljes körű Exploratory Data Analysis (EDA, azaz feltáró
adatelemzés) elvégzése egy használtautó-adathalmazon, kizárólag Suzuki járművekre
fókuszálva, annak érdekében, hogy megértsük:

- Hogyan oszlanak el az árak, és mi befolyásolja őket (életkor, futásteljesítmény,
  váltótípus, tulajdonosi előélet, üzemanyagtípus)
- Mely Suzuki modellek a leggyakoribbak a használtautó-piacon, és hogyan viszonyulnak
  egymáshoz
- Hogyan alakult a modellpaletta és a technológia (üzemanyagtípus, váltótípus) az évek
  során
- Milyen gyakorlatban is hasznosítható insightokat (üzleti következtetéseket) vonhat le
  ebből egy autókereskedés, egy hirdetési piactér vagy egy vásárló

**Megjegyzés az adatforrásról.** Nyilvánosan elérhető, kizárólag Suzuki modelleket
tartalmazó adathalmaz nem áll rendelkezésre a Kaggle-ön. Emiatt ez a projekt egy
**szintetikusan generált adathalmazt** használ, amely a jól ismert Kaggle
*"Car Details Dataset"* sémáját és statisztikai mintázatait tükrözi (használtautó-hirdetések
`name, year, selling_price, km_driven, fuel, seller_type, transmission, owner, mileage,
engine, max_power, seats` oszlopokkal), Suzuki/Maruti Suzuki modellekre szűkítve, reális
értékcsökkenési, futásteljesítmény- és árazási összefüggésekkel feltöltve. Ez lehetővé
teszi, hogy a munkafolyamat, a kód és az elemzési technikák pontosan megegyezzenek
azzal, amit egy valós, lekapart (scraped) adathalmazon alkalmaznánk, miközben biztosítja,
hogy a notebook külső letöltés nélkül, végig lefusson.

Az adatgenerálás logikáját a notebook végén, a teljes átláthatóság és reprodukálhatóság
érdekében szintén közöljük.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42


ModuleNotFoundError: No module named 'matplotlib'

## 2. Az adathalmaz bemutatása

A nyers `suzuki_cars_raw.csv` fájl **Suzuki / Maruti Suzuki használtautó-hirdetéseket**
tartalmaz. Minden sor egy hirdetésnek felel meg. Oszlopok:

| Oszlop | Leírás |
|---|---|
| `name` | A hirdetés teljes neve (márka + modell), pl. "Suzuki Swift" |
| `model` | Csak a modellnév, pl. "Swift", "Baleno", "Ertiga" |
| `segment` | Karosszéria-/szegmenskategória (Hatchback, Sedan, MUV, Compact SUV, SUV) |
| `year` | Gyártási év |
| `selling_price_lakh` | Hirdetett eladási ár, Lakh INR-ben (1 Lakh = 100 000) |
| `km_driven` | Megtett kilométerek száma |
| `fuel` | Üzemanyagtípus (Petrol / benzin, Diesel / dízel, CNG) |
| `seller_type` | Individual (magánszemély), Dealer (kereskedő) vagy Trustmark Dealer (hitelesített kereskedő) |
| `transmission` | Manual (manuális) vagy Automatic (automata) |
| `owner` | Tulajdonosi előélet (First Owner, Second Owner, ...) |
| `mileage_kmpl` | Fogyasztási hatékonyság km/l-ben (az alábbiakban tisztítandó, "szennyezett" szöveges értékeket is tartalmaz) |
| `engine_cc` | Motor lökettérfogata cm³-ben |
| `max_power_bhp` | Maximális teljesítmény bhp-ban |
| `seats` | Ülőhelyek száma |

A nyers fájl szándékosan tartalmaz valósághű adatminőségi problémákat (hiányzó
értékek, duplikált sorok, inkonzisztens szövegformázás, mértékegységgel "szennyezett"
numerikus mezők, valamint néhány kiugró érték / outlier), hogy az alábbi adattisztítási
szakasz egy valódi EDA-munkafolyamatot tükrözzön.


In [ ]:
df_raw = pd.read_csv("suzuki_cars_raw.csv")
print(f"Alakja (shape): {df_raw.shape[0]} sor x {df_raw.shape[1]} oszlop")
df_raw.head()


In [ ]:
df_raw.info()


## 3. Adattisztítás

Sorban a következőket kezeljük:

1. Duplikált sorok
2. Inkonzisztens szövegformázás a kategorikus oszlopokban
3. Vegyes típusú numerikus oszlopok (mértékegység szövegként beágyazva)
4. Hiányzó értékek
5. Nyilvánvaló kiugró értékek (outlierek) / adatrögzítési hibák


In [ ]:
df = df_raw.copy()

# --- 3.1 Duplikátumok -------------------------------------------------
n_dupes = df.duplicated().sum()
print(f"Talált duplikált sorok száma: {n_dupes}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Alakja a duplikátumok eltávolítása után: {df.shape}")


In [ ]:
# --- 3.2 Inkonzisztens szövegformázás --------------------------------
for col in ["fuel", "seller_type", "transmission", "owner", "segment", "model"]:
    df[col] = df[col].astype(str).str.strip().str.title()

print(df["fuel"].unique())
print(df["transmission"].unique())


In [ ]:
# --- 3.3 Vegyes típusú numerikus oszlop: mileage_kmpl ---------------------
# Néhány érték "21.4 kmpl" formában van tárolva egy tiszta float helyett
df["mileage_kmpl"] = (
    df["mileage_kmpl"]
    .astype(str)
    .str.replace("kmpl", "", regex=False)
    .str.strip()
)
df["mileage_kmpl"] = pd.to_numeric(df["mileage_kmpl"], errors="coerce")
print(df["mileage_kmpl"].describe())


## 4. Hiányzó értékek


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"hianyzo_darabszam": missing, "hianyzo_szazalek": missing_pct})
missing_summary = missing_summary[missing_summary["hianyzo_darabszam"] > 0].sort_values("hianyzo_darabszam", ascending=False)
missing_summary


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
if len(missing_summary):
    sns.barplot(x=missing_summary.index, y=missing_summary["hianyzo_szazalek"], ax=ax, color="#e07a5f")
    ax.set_ylabel("Hiányzó érték (%)")
    ax.set_xlabel("")
    ax.set_title("Hiányzó értékek oszloponként")
    for i, v in enumerate(missing_summary["hianyzo_szazalek"]):
        ax.text(i, v + 0.05, f"{v}%", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


In [3]:
# A numerikus oszlopokat modellenkénti (model) mediánnal töltjük fel,
# ami pontosabb, mint egyetlen globális medián egy több modellt tartalmazó adathalmaznál.
numeric_cols_to_impute = ["mileage_kmpl", "engine_cc", "max_power_bhp", "seats"]

for col in numeric_cols_to_impute:
    df[col] = df.groupby("model")[col].transform(lambda s: s.fillna(s.median()))
    # tartalék: ha egy ritka modellnél így is maradna hiányzó érték, globális mediánnal töltjük fel
    df[col] = df[col].fillna(df[col].median())

df["seats"] = df["seats"].round().astype(int)

print("Fennmaradó hiányzó értékek:")
print(df[numeric_cols_to_impute].isnull().sum())


NameError: name 'df' is not defined

## 5. Adattípusok

A tisztítás után az oszlopokat a megfelelő típusra alakítjuk: a numerikus mezőket
`float`/`int` típusra, a kategorikus mezőket pandas `category` típusra (memóriahatékony
és szemantikailag helyes megoldás ismétlődő szöveges értékek esetén), és létrehozunk
néhány kényelmi segédoszlopot (`car_age`; a `price_per_100k_km`-hez hasonlók ott kerülnek
bevezetésre az elemzésben, ahol szükségesek).


In [ ]:
categorical_cols = ["model", "segment", "fuel", "seller_type", "transmission", "owner"]
for col in categorical_cols:
    df[col] = df[col].astype("category")

df["year"] = df["year"].astype(int)
df["km_driven"] = df["km_driven"].astype(int)
df["engine_cc"] = df["engine_cc"].round().astype(int)

df["car_age"] = 2024 - df["year"]

df.dtypes


In [ ]:
# --- 3.5 Kiugró értékek (outlierek) ------------------------------------------------------
# Extrém km_driven és irreálisan alacsony árak jelölése az IQR-szabály alapján
def iqr_bounds(series, k=3.0):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

km_low, km_high = iqr_bounds(df["km_driven"])
price_low, price_high = iqr_bounds(df["selling_price_lakh"])

outlier_mask = (df["km_driven"] > km_high) | (df["selling_price_lakh"] < max(price_low, 0.3))
print(f"Detektált outlier sorok száma: {outlier_mask.sum()}")
df.loc[outlier_mask, ["name", "year", "km_driven", "selling_price_lakh"]]


In [ ]:
df_clean = df.loc[~outlier_mask].reset_index(drop=True)
print(f"A végleges, megtisztított adathalmaz alakja: {df_clean.shape}")
df_clean.head()


## 6. Leíró statisztikák


In [ ]:
df_clean.describe(include="number").T


In [ ]:
df_clean.describe(include="category").T


## 7. Eloszláselemzés

Megvizsgáljuk a legfontosabb numerikus változók eloszlásának alakját: az ár, a
futásteljesítmény, a fogyasztás, a motorméret és az autó életkora.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
num_vars = [
    ("selling_price_lakh", "Eladási ár (Lakh INR)"),
    ("km_driven", "Futásteljesítmény (km)"),
    ("mileage_kmpl", "Fogyasztás (km/l)"),
    ("engine_cc", "Motor lökettérfogata (cm³)"),
    ("car_age", "Az autó életkora (év)"),
    ("max_power_bhp", "Maximális teljesítmény (bhp)"),
]

for ax, (col, label) in zip(axes.flat, num_vars):
    sns.histplot(df_clean[col], kde=True, ax=ax, color="#3d5a80")
    ax.set_title(label)
    ax.set_xlabel("")
    skew = df_clean[col].skew()
    ax.text(0.97, 0.92, f"ferdeség={skew:.2f}", transform=ax.transAxes, ha="right", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.7))

plt.suptitle("A legfontosabb numerikus változók eloszlása", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


**Az eloszlások értelmezése:**
- A `selling_price_lakh` jobbra ferde (right-skewed): a Suzuki-hirdetések többsége a
  megfizethető belépő/közepes kategóriában található, hosszú "farokkal" a prémium
  modellek felé (Grand Vitara, Jimny, XL6).
- A `km_driven` szintén jobbra ferde, amit néhány magasabb futásteljesítményű, idősebb
  autó okoz.
- A `mileage_kmpl` egy **kvázi bimodális** eloszlást mutat, ami a benzines és CNG
  változatok keverékét tükrözi (a CNG autók jellemzően jóval magasabb km/l értéket
  jelentenek).
- A `car_age` nagyjából a használtautó-hirdetések mintavételi időablakát tükrözi.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(x=df_clean["selling_price_lakh"], ax=axes[0], color="#98c1d9")
axes[0].set_title("Eladási ár — Boxplot (dobozdiagram)")
sns.boxplot(x=df_clean["km_driven"], ax=axes[1], color="#ee6c4d")
axes[1].set_title("Futásteljesítmény — Boxplot (dobozdiagram)")
plt.tight_layout()
plt.show()


## 8. Árelemzés

Itt azt vizsgáljuk, hogyan függ össze az eladási ár az életkorral, a futásteljesítménnyel,
a váltótípussal és a tulajdonosi előélettel — ezek a klasszikus értékcsökkenési tényezők
a használtautó-piacon.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df_clean, x="car_age", y="selling_price_lakh", hue="segment",
                 alpha=0.6, ax=ax, palette="Set2")
ax.set_title("Eladási ár vs. az autó életkora, szegmensenként")
ax.set_xlabel("Az autó életkora (év)")
ax.set_ylabel("Eladási ár (Lakh INR)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Szegmens")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(data=df_clean, x="km_driven", y="selling_price_lakh",
            scatter_kws={"alpha": 0.25, "s": 20}, line_kws={"color": "#e07a5f"}, ax=ax)
ax.set_title("Eladási ár vs. futásteljesítmény")
ax.set_xlabel("Futásteljesítmény (km)")
ax.set_ylabel("Eladási ár (Lakh INR)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}e"))
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df_clean, x="transmission", y="selling_price_lakh", ax=axes[0], palette="pastel")
axes[0].set_title("Ár váltótípus szerint")
axes[0].set_xlabel("Váltótípus")
axes[0].set_ylabel("Eladási ár (Lakh INR)")

owner_order = ["First Owner", "Second Owner", "Third Owner", "Fourth & Above Owner"]
owner_labels_hu = ["1. tulajdonos", "2. tulajdonos", "3. tulajdonos", "4. vagy több tulajdonos"]
sns.boxplot(data=df_clean, x="owner", y="selling_price_lakh", order=owner_order, ax=axes[1], palette="pastel")
axes[1].set_title("Ár tulajdonosi előélet szerint")
axes[1].set_xlabel("Tulajdonosi előélet")
axes[1].set_ylabel("")
axes[1].set_xticklabels(owner_labels_hu)
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
avg_price_by_owner = df_clean.groupby("owner", observed=True)["selling_price_lakh"].mean().reindex(owner_order)
avg_price_by_trans = df_clean.groupby("transmission", observed=True)["selling_price_lakh"].mean()

print("Átlagár tulajdonosi előélet szerint:")
print(avg_price_by_owner.round(2))
print("\nÁtlagár váltótípus szerint:")
print(avg_price_by_trans.round(2))


**Legfontosabb megfigyelések:**
- Az ár folyamatosan csökken az **autó életkorával**, ami megfelel a szokásos
  értékcsökkenési görbének.
- A magasabb **km_driven** (futásteljesítmény) alacsonyabb árral jár együtt, bár ez az
  összefüggés zajosabb, mint önmagában az életkor — a használat intenzitása számít, de
  a modell/állapot is.
- Az **automata váltós** Suzuki autók egyértelmű árprémiumot élveznek a hasonló korú
  manuális változatokhoz képest, ami azt tükrözi, hogy az AMT/AT modellek relatíve
  újabbak és magasabb felszereltségi szintűek.
- Az ár monoton csökken, ahogy az **1. tulajdonostól** a **4. vagy több tulajdonos**
  felé haladunk, megerősítve, hogy a tulajdonosi előélet erős és könnyen ellenőrizhető
  árjelzés.


## 9. Modellösszehasonlítás

A Suzuki modellpalettája hatchbackeket, szedánokat, MUV-okat és SUV-okat egyaránt
felölel. Összehasonlítjuk a hirdetések számát, az átlagárat és az árszórást modellenként.


In [ ]:
model_counts = df_clean["model"].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=model_counts.values, y=model_counts.index, ax=ax, palette="viridis")
ax.set_title("Hirdetések száma modellenként")
ax.set_xlabel("Hirdetések száma")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
order_by_price = df_clean.groupby("model", observed=True)["selling_price_lakh"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_clean, x="selling_price_lakh", y="model", order=order_by_price, ax=ax, palette="coolwarm")
ax.set_title("Eladási ár eloszlása modellenként")
ax.set_xlabel("Eladási ár (Lakh INR)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


In [4]:
model_summary = df_clean.groupby("model", observed=True).agg(
    hirdetesek_szama=("model", "count"),
    atlagar=("selling_price_lakh", "mean"),
    median_ar=("selling_price_lakh", "median"),
    atlag_futasteljesitmeny=("km_driven", "mean"),
    atlag_eletkor=("car_age", "mean"),
).round(2).sort_values("atlagar", ascending=False)

model_summary


NameError: name 'df_clean' is not defined

**Legfontosabb megfigyelések:**
- A **Swift** és a **Dzire** dominálja a hirdetési volument — ez nem meglepő, mivel
  történelmileg ezek a Suzuki legkelendőbb típusai.
- A prémium/SUV modellek (**Grand Vitara, Jimny, XL6, Ciaz**) érik el a legmagasabb
  medián eladási árat, ugyanakkor jóval ritkábban fordulnak elő, ami az alacsonyabb
  eladási volument és/vagy a frissebb piaci bevezetést tükrözi.
- A belépő szintű hatchbackek (**Alto K10, S-Presso, Celerio**) horgonyozzák le a
  piac megfizethető szegmensét, ahogy az várható volt.


## 10. Éves trendek

Hogyan változott a Suzuki hirdetési mixe — szegmens-népszerűség, üzemanyagtípus,
váltótípus — a gyártási évek mentén?


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
year_counts = df_clean["year"].value_counts().sort_index()
sns.lineplot(x=year_counts.index, y=year_counts.values, marker="o", ax=ax, color="#3d5a80")
ax.set_title("Hirdetések száma gyártási év szerint")
ax.set_xlabel("Év")
ax.set_ylabel("Hirdetések száma")
plt.tight_layout()
plt.show()


In [ ]:
year_price = df_clean.groupby("year", observed=True)["selling_price_lakh"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(x=year_price.index, y=year_price.values, marker="o", ax=ax, color="#e07a5f")
ax.set_title("Átlagos eladási ár gyártási év szerint")
ax.set_xlabel("Év")
ax.set_ylabel("Átlagos eladási ár (Lakh INR)")
plt.tight_layout()
plt.show()


In [ ]:
trans_by_year = pd.crosstab(df_clean["year"], df_clean["transmission"], normalize="index") * 100

fig, ax = plt.subplots(figsize=(10, 5))
trans_by_year.plot(kind="area", stacked=True, ax=ax, colormap="Set2", alpha=0.85)
ax.set_title("Automata vs. manuális arány az idő függvényében")
ax.set_ylabel("Hirdetések aránya (%)")
ax.set_xlabel("Év")
ax.legend(title="Váltótípus", loc="upper left")
plt.tight_layout()
plt.show()


**Legfontosabb megfigyelések:**
- Az átlagos eladási ár emelkedik a gyártási évvel, ami főként az **alacsonyabb
  életkort** (kevesebb értékcsökkenést) tükrözi az újabb hirdetéseknél, nem pedig az
  eredeti listaár változását.
- Az **automata váltós** hirdetések aránya folyamatosan nő az újabb gyártási éveknél —
  ez összhangban van azzal, hogy a Suzuki a 2010-es évek közepétől kezdve egyre több
  modelljénél kínál AMT/CVT/hagyományos automata opciót.


## 11. Fogyasztáselemzés

A "mileage" (fogyasztás/hatékonyság) itt az üzemanyag-hatékonyságra (km/l) utal —
ez az egyik kulcsfontosságú vásárlási szempont abban a megfizethető/kompakt
szegmensben, ahol a Suzuki versenyez.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df_clean, x="fuel", y="mileage_kmpl", ax=ax, palette="Set2")
ax.set_title("Üzemanyag-hatékonyság (km/l) üzemanyagtípus szerint")
ax.set_xlabel("Üzemanyagtípus")
ax.set_ylabel("Fogyasztás (km/l)")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
mileage_order = df_clean.groupby("model", observed=True)["mileage_kmpl"].mean().sort_values(ascending=False).index
sns.barplot(data=df_clean, y="model", x="mileage_kmpl", order=mileage_order, ax=ax, palette="crest", errorbar=None)
ax.set_title("Átlagos fogyasztás modellenként")
ax.set_xlabel("Átlagos fogyasztás (km/l)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
corr_mileage_km, p_val = stats.pearsonr(df_clean["km_driven"], df_clean["mileage_kmpl"])
print(f"Korreláció a km_driven és a mileage_kmpl között: r = {corr_mileage_km:.3f} (p = {p_val:.3g})")


**Legfontosabb megfigyelések:**
- A **CNG változatok** érzékelhetően magasabb km/l értéket jelentenek, mint a benzines
  társaik, a dízel (ami itt csak MUV/SUV modelleknél fordul elő) pedig a benzin és a
  CNG között helyezkedik el.
- A kompakt hatchbackek (Celerio, WagonR, S-Presso, Alto K10) vezetik az átlagos
  fogyasztási rangsort, összhangban kisebb, könnyebb platformjukkal — míg az SUV-ok,
  mint a Grand Vitara és a Jimny, lemaradnak, a nagyobb motor és önsúly miatt, ahogy
  az várható is volt.
- A `km_driven` és a `mileage_kmpl` közötti korreláció gyenge/elhanyagolható, ami azt
  jelenti, hogy az üzemanyag-hatékonyság elsősorban a modelltől/üzemanyagtípustól függ,
  nem a használat mértékétől.


## 12. Üzemanyagtípus szerinti elemzés


In [ ]:
fuel_counts = df_clean["fuel"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(fuel_counts.values, labels=fuel_counts.index, autopct="%1.1f%%",
            colors=sns.color_palette("Set2"), startangle=90)
axes[0].set_title("Üzemanyagtípusok megoszlása")

sns.boxplot(data=df_clean, x="fuel", y="selling_price_lakh", ax=axes[1], palette="Set2")
axes[1].set_title("Eladási ár üzemanyagtípus szerint")
axes[1].set_xlabel("Üzemanyagtípus")
axes[1].set_ylabel("Eladási ár (Lakh INR)")

plt.tight_layout()
plt.show()


In [ ]:
fuel_segment = pd.crosstab(df_clean["segment"], df_clean["fuel"])
fuel_segment


**Legfontosabb megfigyelések:**
- A **benzin (Petrol)** dominálja a Suzuki hirdetési mixet, összhangban a Suzuki
  kisautó-központú, elsősorban benzines stratégiájával (különösen releváns az indiai
  piacon, ahol piacvezető).
- A **CNG** változatok jelentős kisebbséget alkotnak, elsősorban a hatchbackekben és
  szedánokban koncentrálódva, ahol a gyári CNG-opciók elterjedtek (Swift, WagonR,
  Dzire, Celerio).
- A **dízel** csak néhány MUV hirdetésnél jelenik meg, ami az iparág — és a Suzuki —
  szélesebb körű elmozdulását tükrözi a dízelmotoroktól a kisautó-kategóriában.


## 13. Váltótípus szerinti elemzés


In [ ]:
trans_counts = df_clean["transmission"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(trans_counts.values, labels=trans_counts.index, autopct="%1.1f%%",
            colors=["#98c1d9", "#ee6c4d"], startangle=90)
axes[0].set_title("Váltótípusok megoszlása")

sns.violinplot(data=df_clean, x="transmission", y="selling_price_lakh", ax=axes[1], palette=["#98c1d9", "#ee6c4d"])
axes[1].set_title("Ár eloszlása váltótípus szerint")
axes[1].set_xlabel("Váltótípus")
axes[1].set_ylabel("Eladási ár (Lakh INR)")

plt.tight_layout()
plt.show()


In [ ]:
trans_by_segment = pd.crosstab(df_clean["segment"], df_clean["transmission"], normalize="index") * 100
trans_by_segment.round(1)


**Legfontosabb megfigyelések:**
- A **manuális** váltó továbbra is a hirdetések többségét adja, ami a Suzuki
  költséghatékony, hagyományosan manuális örökségét tükrözi a belépő szintű
  szegmensekben.
- Az **automata** váltó aránya a **Compact SUV / SUV / MUV** szegmensekben a
  legmagasabb, ahol a vásárlók hajlandóbbak fizetni a kényelmi prémiumért.
- Az automata hirdetések — ahogy korábban is láttuk — érezhetően magasabb medián árat
  mutatnak, ami egyszerre felszereltségi szint és életkor hatás (az automaták
  átlagosan újabbak).


## 14. Korrelációelemzés


In [ ]:
num_cols = ["selling_price_lakh", "km_driven", "year", "car_age", "mileage_kmpl", "engine_cc", "max_power_bhp", "seats"]
corr = df_clean[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, linewidths=0.5)
ax.set_title("Korrelációs mátrix — numerikus változók")
plt.tight_layout()
plt.show()


In [ ]:
price_corr = corr["selling_price_lakh"].drop("selling_price_lakh").sort_values(key=abs, ascending=False)
price_corr


**Legfontosabb megfigyelések:**
- A `car_age` (és fordítottan a `year`) az eladási ár **legerősebb meghatározója** —
  ez megerősíti, hogy az értékcsökkenés a domináns erő ezen a piacon, ahogy az a
  használtautó-piacokon jellemző.
- A `max_power_bhp` és az `engine_cc` pozitívan korrelál az árral, mivel a magasabb
  kategóriájú Suzuki modellek (SUV/MUV) egyszerre drágábbak és nagyobb motorral
  rendelkeznek.
- A `km_driven` negatívan korrelál az árral, de gyengébben, mint az életkor — két
  azonos korú autónak nagyon eltérő lehet a használati előélete, ami zajt visz az
  összefüggésbe.
- A `mileage_kmpl` enyhe negatív korrelációt mutat az árral, ez főként abból adódik,
  hogy a legfogyasztás-hatékonyabb autók egyben a legolcsóbb belépő szintű
  hatchbackek is — nem pedig abból, hogy a hatékonyság önmagában csökkentené az
  értéket.


## 15. Üzleti következtetések

A fenti szakaszokat összefogva, egy autókereskedés, egy használtautó-piactér vagy egy
Suzuki-fókuszú elemző szemszögéből:

1. **Az értékcsökkenés kiszámítható és életkor-vezérelt.** Mivel a `car_age` messze a
   legerősebb árkorreláció, egy egyszerű, életkor-alapú árazási alapmodell (a
   tulajdonosszám és a váltótípus figyelembevételével korrigálva) már önmagában is
   megmagyarázná az árszórás nagy részét — ez hasznos első lépésű
   értékbecslési modellként szolgálhat, mielőtt az állapotot és a piaci keresletet is
   beépítenénk.

2. **A tulajdonosi előélet egy egyszerű, magas hozzáadott értékű árazási eszköz.** Az
   egyértelmű árkülönbség az 1. tulajdonosú és a későbbi tulajdonosú autók között azt
   sugallja, hogy a piactereknek kiemelten kellene megjeleníteniük a tulajdonosszámot,
   a kereskedőknek pedig priorizálniuk kellene az alacsony tulajdonosszámú készletek
   beszerzését, amelyek prémiumáron kelnek el.

3. **Az automata változatok alulreprezentáltak, de prémiumot élveznek**, különösen a
   Compact SUV/SUV/MUV szegmensekben. Ez lehetőséget jelez: az automata váltós Vitara
   Brezza, Fronx vagy Ertiga egységeket készleten tartó kereskedők gyorsabb
   forgást és jobb árrést érhetnek el, ahogy a vásárlói preferenciák a kényelem felé
   tolódnak.

4. **A CNG-változatok tovább erősítik a Suzuki fogyasztási előnyét.** Tekintve a
   Suzuki erős CNG-penetrációját a hatchbackek/szedánok körében, ezen változatok
   marketingjében az üzemeltetési költségmegtakarítás hangsúlyozása (nem csak a
   listaár) megkülönböztető tényező lehet a gyengébb CNG-palettával rendelkező
   versenytársakhoz képest.

5. **Volumen vs. árrés kompromisszum a modellkínálaton belül.** A Swift/Dzire uralja a
   volument (hasznos egy forgás-vezérelt stratégiához), míg a Grand Vitara/Jimny/XL6
   magasabb egységenkénti árrést kínál, de lassabb forgás mellett — ez egy
   portfólió-kiegyensúlyozási szempont a korlátozott készlettel dolgozó kereskedők
   számára.

6. **A km_driven önmagában hiányos árjelzés.** Mivel a használat intenzitása
   olyannyira eltér az azonos korú autók között, az értékbecslési modelleknek (és a
   vásárlóknak) a magas futásteljesítményt egy életkor-alapú alapérték másodlagos
   korrekciójaként kellene kezelniük, nem elsődleges meghatározó tényezőként.


## 16. Záró következtetések

A Suzuki használtautó-szegmens ezen EDA-ja egy elsősorban **életkor-vezérelt
értékcsökkenés** által meghatározott piacot mutat, ahol a **tulajdonosi előélet** és a
**váltótípus** másodlagos, de érdemi árbefolyásoló tényezők. A Suzuki termékmixe —
jellemzően benzin/CNG-alapú, hatchback- és szedán-vezérelt, egyre növekvő, de még
mindig rés jellegű SUV/MUV jelenléttel — jól látszik az adatokban: a volumen a
megfizethető, fogyasztás-hatékony modelleknél (Swift, Dzire, WagonR) koncentrálódik,
míg az árplafont az újabb SUV/MUV modellek (Grand Vitara, Jimny, XL6) határozzák meg.

**Korlátok:** ez a notebook egy *szintetikusan generált* adathalmazt használ, amelyet
úgy alakítottunk ki, hogy reális használtautó-piaci összefüggéseket tükrözzön, nem
valós, lekapart (scraped) hirdetéseket. A minőségi mintázatok (értékcsökkenési görbék,
CNG fogyasztási előny, automata váltós prémium, modell-szegmens szerinti árhierarchia)
azon alapulnak, ahogyan ezek a piacok a valóságban működnek, de a pontos számokat nem
szabad valós piaci adatként kezelni. A projekt kiterjesztése valódi Suzuki
hirdetésekkel (pl. egy piactér exportjából vagy egy Suzuki/Maruti Suzuki modellekre
szűrt Kaggle adathalmazból) lehetővé tenné, hogy ugyanez a feldolgozási folyamat
éles minőségű insightokat produkáljon.

**Javasolt következő lépések:**
- Egy alap árelőrejelző modell (pl. gradient boosting) illesztése a `car_age`,
  `km_driven`, `transmission`, `owner` és `segment` jellemzők (feature-ök)
  felhasználásával, és összevetése a korrelációelemzésből adódó egyszerű,
  életkor-alapú heurisztikával.
- Valós regionális árazási adatok bevonása, ha egy konkrét használtautó-piacot céloz
  meg az elemzés.
- A hirdetésszám és az árak alakulásának időbeli (pl. havi) nyomon követése, ha élő
  piactéri adatfolyammal, nem pedig egy statikus pillanatképpel dolgozunk.


---
### Függelék: Adatgenerálási logika

Az átláthatóság érdekében: a notebookban használt szintetikus adathalmaz a következő
logikával készült (modellspecifikus alapárak, szegmens-alapú fogyasztási/motor
profilok, exponenciális, életkor szerinti értékcsökkenési görbe, valamint a
`km_driven`/tulajdonosi előélet/váltótípus szerinti árkorrekció), majd erre valósághű
"szennyeződéseket" (duplikátumok, hiányzó értékek, inkonzisztens szövegformázás,
mértékegységgel beágyazott szöveges értékek és outlierek) helyeztünk rá — a teljes,
reprodukálható script a projektfájlok között a `generate_data.py` fájlban található.
